# CS383 — Assignment 3
## Building and Judging Models: Regression and Classification with scikit-learn Datasets

*Due: see LMS*

### Scenario
A small hospital research group has asked you, as a data intern, to try out two simple prediction tools
on well-known medical datasets that ship with scikit-learn:

1. **Diabetes progression** — can a linear regression predict how far a patient's disease has progressed
   one year after their baseline measurements?
2. **Tumor diagnosis** — can logistic regression and k-NN tell malignant from benign tumors using
   measurements taken from a cell sample?

This is the skill Lectures 7 and 8 built toward: fitting the models, then judging them honestly
against a baseline instead of trusting a single number. Both datasets are built into scikit-learn, so
there is nothing to download.

---

### Academic integrity and using AI

This assignment allows AI-assisted coding, the same way earlier assignments did — but the line is about *what* AI is allowed to do, not *whether* you can use it at all.

**Allowed** — using an AI tool (Claude, ChatGPT, GitHub Copilot, etc.) to help write the *mechanical* code in each deliverable's code cell: splitting data, fitting a model, computing a metric, making a table.

**Not allowed** — having AI write your reasoning, interpretation, or written answers. Every written answer below (what a metric means, which model is better, which mistake is worse, why you shouldn't read a coefficient as a cause) is the actual judgment this assignment is testing. That judgment has to be yours.

**If you use AI for a code cell, add a one-line comment directly above it naming the prompt**, for example:

```python
# AI prompt: "fit a LogisticRegression inside a pipeline with StandardScaler and report test accuracy"
model = ...
```

Only comment the lines that AI actually helped with.

---

### How this notebook works

Run the cell below once. It loads both datasets and a small helper function. **Do not modify it** — every
deliverable assumes these variables exist.

- `X_diab`, `y_diab`: 442 patients, 10 features (age, sex, bmi, bp, and six blood-serum measurements
  `s1`–`s6`), and a numeric target measuring disease progression one year later. scikit-learn has already
  centered and scaled these ten features.
- `X_cancer`, `y_cancer`: 569 tumors, 30 measurement features, and a 0/1 target.

Every deliverable has the same three pieces:

1. **A short task.**
2. **A code cell where most of the code is already written.** You only write the lines under
   `YOUR CODE`; the parts marked `PROVIDED` (tables and charts) run on what you wrote, so don't change them.
3. **A written answer.** The charts are there to help you answer — look at them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix, precision_score, recall_score,
                             ConfusionMatrixDisplay)

diabetes = load_diabetes(as_frame=True)
X_diab, y_diab = diabetes.data, diabetes.target

cancer = load_breast_cancer(as_frame=True)
X_cancer, y_cancer = cancer.data, cancer.target

def regression_metrics(y_true, y_pred):
    """Return MAE, RMSE and R2 for a set of predictions."""
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

print("Diabetes:", X_diab.shape, "| target:", y_diab.name)
print("Breast cancer:", X_cancer.shape, "| target classes:", [str(n) for n in cancer.target_names])

---

### Look at the data first

Run the cell below to see both datasets: how disease progression changes with `bmi`, how many tumors of each
class there are, and how two of the tumor features separate the classes. Then use the scratch cell for
anything else you want to try (for example `X_diab.describe()`). Note that in the cancer data
**class 0 = malignant and class 1 = benign** (`cancer.target_names` shows the order).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Diabetes: one feature vs. the target
axes[0].scatter(X_diab["bmi"], y_diab, s=12, alpha=0.6)
axes[0].set(xlabel="bmi (scaled)", ylabel="disease progression", title="Diabetes: bmi vs. progression")

# Breast cancer: how many tumors of each class?
counts = y_cancer.value_counts().sort_index()
axes[1].bar(cancer.target_names[counts.index], counts.values, color=["crimson", "steelblue"])
axes[1].set(ylabel="number of tumors", title="Breast cancer: class counts")

# Breast cancer: two features, colored by class
for cls, color in [(0, "crimson"), (1, "steelblue")]:
    mask = y_cancer == cls
    axes[2].scatter(X_cancer.loc[mask, "mean area"], X_cancer.loc[mask, "mean smoothness"],
                    s=12, alpha=0.6, color=color, label=cancer.target_names[cls])
axes[2].set(xlabel="mean area", ylabel="mean smoothness", title="Breast cancer: two features by class")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Scratch space (optional) -- try anything else here.

---

## Part 1 — Regression on the Diabetes Data

### Required Deliverable 1 — Split and Baseline (10 pts)

**The code for this deliverable is provided — run the cell below and read its output and chart.** It splits the
diabetes data 80/20 (`random_state=383`) into `Xd_train`, `Xd_test`, `yd_train`, `yd_test`, then builds the
**baseline**: a "model" that ignores every feature and predicts the **mean of `yd_train`** for every test patient.
Later deliverables reuse these variables.

**Answer in 1–2 sentences each:**

1. R² compares a model against the simple guess "always predict the **test** set's mean." Look at the two means
   printed below. Why is the baseline's R² slightly below 0 instead of exactly 0?
2. What is this baseline for? What does it give you to compare your real model against?

In [ ]:
# --- PROVIDED: just run this cell ---------------------------------------------
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_diab, y_diab, test_size=0.2, random_state=383)

# Baseline: ignore every feature; predict the TRAINING mean for every test patient
train_mean = yd_train.mean()
baseline_pred = np.full(len(yd_test), train_mean)
baseline_scores = regression_metrics(yd_test, baseline_pred)

print(f"Training mean (the baseline's prediction): {train_mean:.1f}")
print(f"Test mean (what R2 is measured against):   {yd_test.mean():.1f}")
print(f"Baseline test scores:  MAE {baseline_scores['MAE']:.1f} | RMSE {baseline_scores['RMSE']:.1f} | R2 {baseline_scores['R2']:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(yd_test, bins=15, color="lightsteelblue", edgecolor="white")
ax.axvline(train_mean, color="crimson", lw=2, label=f"baseline prediction = training mean ({train_mean:.1f})")
ax.axvline(yd_test.mean(), color="black", ls="--", lw=2, label=f"test mean ({yd_test.mean():.1f})")
ax.set(xlabel="disease progression (test patients)", ylabel="number of patients", ylim=(0, 14),
       title="The baseline predicts one number for everyone")
ax.legend()
plt.tight_layout()
plt.show()

**Your answers** (1. why R² is slightly below 0; 2. what the baseline is for):

---

### Required Deliverable 2 — Linear Regression (20 pts)

**Task.** Fit two `LinearRegression` models on the **training** data:

- **Model A (`model_a`):** all 10 features.
- **Model B (`model_b`):** only the `bmi` feature (use `Xd_train[["bmi"]]` with double brackets).

The provided code then evaluates both on the test set and shows a table (with the baseline), an
actual-vs-predicted chart, a `bmi` chart, and a coefficient chart.

**Answer:**

1. Which model predicts better, and by roughly how much does it improve on the baseline? In plain words, what
   does Model A's R² tell you about how much of the variation in disease progression it explains?
2. In Model A, which feature has the **largest positive coefficient**? Give one reason why you should **not**
   conclude that changing that feature would change a patient's disease progression.

In [ ]:
# --- YOUR CODE (2 lines) ----------------------------------------------------
# 1. model_a: a LinearRegression fitted on Xd_train (all 10 features) and yd_train
# 2. model_b: a LinearRegression fitted on Xd_train[["bmi"]] and yd_train


# --- PROVIDED: do not change ------------------------------------------------------
pred_a = model_a.predict(Xd_test)
pred_b = model_b.predict(Xd_test[["bmi"]])

results = pd.DataFrame({
    "Baseline (training mean)": regression_metrics(yd_test, baseline_pred),
    "A: all 10 features":       regression_metrics(yd_test, pred_a),
    "B: bmi only":              regression_metrics(yd_test, pred_b),
}).T.round(2)
print(results)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# (1) Actual vs. predicted for Model A: points on the diagonal = perfect predictions
axes[0].scatter(yd_test, pred_a, s=15, alpha=0.7)
lims = [yd_test.min(), yd_test.max()]
axes[0].plot(lims, lims, "k--", label="perfect prediction")
axes[0].axhline(train_mean, color="crimson", ls=":", label="baseline")
axes[0].set(xlabel="actual progression", ylabel="predicted progression", title="Model A: actual vs. predicted")
axes[0].legend()

# (2) Model B: the fitted line through the bmi data
order = np.argsort(Xd_test["bmi"].values)
axes[1].scatter(Xd_test["bmi"], yd_test, s=15, alpha=0.6, label="test patients")
axes[1].plot(Xd_test["bmi"].values[order], pred_b[order], color="darkorange", lw=2, label="Model B (bmi only)")
axes[1].axhline(train_mean, color="crimson", ls=":", label="baseline")
axes[1].set(xlabel="bmi (scaled)", ylabel="disease progression", title="Model B: one feature, one line")
axes[1].legend()

# (3) Model A coefficients
coefs = pd.Series(model_a.coef_, index=X_diab.columns).sort_values()
axes[2].barh(coefs.index, coefs.values, color=["crimson" if v < 0 else "steelblue" for v in coefs.values])
axes[2].axvline(0, color="black", lw=0.8)
axes[2].set(xlabel="coefficient", title="Model A coefficients")

plt.tight_layout()
plt.show()
print(coefs.round(1))

**Your answer** (1. which model is better and what R² means; 2. the largest coefficient and why it is not a cause):

---

## Part 2 — Classification on the Breast Cancer Data

### Required Deliverable 3 — Split, Baseline, and Logistic Regression (15 pts)

**Task.** Write the lines described under `YOUR CODE`: split the data 80/20 (stratified), compute the
**majority-class baseline** accuracy, and fit a **logistic regression inside a pipeline with `StandardScaler`**.
The provided code prints the results and charts them.

**Answer in 1–2 sentences each:**

1. How much better than the baseline is the logistic regression?
2. Why is it important that the scaling is done inside the pipeline instead of scaling all of `X_cancer`
   before splitting?

In [ ]:
# --- YOUR CODE (4 steps) ----------------------------------------------------------
# 1. Split X_cancer and y_cancer 80/20 with random_state=383 and stratify=y_cancer
#    -> Xc_train, Xc_test, yc_train, yc_test
# 2. cancer_baseline: the accuracy on the TEST set of always predicting the majority class of yc_train
#    (hint: yc_train.mode()[0] is that class)
# 3. log_pipe: a pipeline of StandardScaler() + LogisticRegression(max_iter=1000), fitted on the training data
# 4. logreg_acc: log_pipe's accuracy on the test set


# --- PROVIDED: do not change ------------------------------------------------------
print(f"Majority-class baseline accuracy: {cancer_baseline:.3f}")
print(f"Logistic regression accuracy:     {logreg_acc:.3f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["majority-class\nbaseline", "logistic\nregression"], [cancer_baseline, logreg_acc],
              color=["lightgray", "steelblue"])
ax.bar_label(bars, fmt="%.3f")
ax.set(ylim=(0, 1.1), ylabel="test accuracy", title="Test accuracy")
plt.tight_layout()
plt.show()

**Your answers** (1. improvement over the baseline; 2. why scaling goes inside the pipeline):

---

### Required Deliverable 4 — k-NN, Scaling, and Choosing k (20 pts)

**Task.** Fit **k-NN** models (`KNeighborsClassifier`) on the training data from Deliverable 3 and record their
**test accuracy**:

- **scaled** k-NN (a pipeline with `StandardScaler`) for each of `k = 1, 5, 15, 25`
- **unscaled** k-NN (no scaler) for `k = 5` only, for comparison

The provided code builds the table and two charts: accuracy vs. `k`, and the raw scale of each feature.

**Answer:**

1. Compare k-NN with your logistic regression from Deliverable 3. Did the more flexible model do better?
2. Look at the right-hand chart (feature scales). Explain **why unscaled k-NN is a risky idea** for this data, even
   if its accuracy on this particular split happens to look fine.
3. A classmate says, "I'll just pick whichever `k` has the highest test accuracy." Why is that a problem, and what
   is a better way to choose `k`?

In [ ]:
# --- YOUR CODE ----------------------------------------------------------------------
knn_scaled_acc = {}                      # will map k -> test accuracy
for k in [1, 5, 15, 25]:
    # Build a pipeline of StandardScaler() + KNeighborsClassifier(n_neighbors=k), fit it on the
    # training data, and store its test accuracy in knn_scaled_acc[k].
    pass                                 # replace this line with your code

# knn_unscaled_acc: test accuracy of a plain KNeighborsClassifier(n_neighbors=5) (no scaler), fitted on the training data


# --- PROVIDED: do not change ------------------------------------------------------
table = {f"scaled k-NN, k={k}": acc for k, acc in knn_scaled_acc.items()}
table["UNSCALED k-NN, k=5"] = knn_unscaled_acc
table["logistic regression (D3)"] = logreg_acc
print(pd.Series(table, name="test accuracy").round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# (1) Accuracy vs. k
ks = list(knn_scaled_acc)
axes[0].plot(ks, list(knn_scaled_acc.values()), "o-", label="scaled k-NN")
axes[0].scatter([5], [knn_unscaled_acc], marker="*", s=220, color="crimson", zorder=3, label="UNSCALED k-NN (k=5)")
axes[0].axhline(logreg_acc, color="gray", ls="--", label="logistic regression")
axes[0].axhline(cancer_baseline, color="black", ls=":", label="majority baseline")
axes[0].set(xlabel="k (number of neighbors)", ylabel="test accuracy", ylim=(0.4, 1.02), xticks=ks,
            title="k-NN test accuracy vs. k")
axes[0].legend(loc="lower right")

# (2) How different are the raw feature scales?
spread = X_cancer.std().sort_values()
axes[1].barh(spread.index, spread.values, color="slategray")
axes[1].set_xscale("log")
axes[1].set(xlabel="standard deviation of the raw feature (log scale)", title="Raw feature scales")
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.show()

**Your answers** (1. k-NN vs. logistic regression; 2. why unscaled is risky; 3. how to choose k):

---

### Required Deliverable 5 — Which Mistakes Matter? (25 pts)

Accuracy treats every mistake the same. Here they are not the same.

**Task.** Using your `log_pipe` from Deliverable 3, compute the test-set predictions, the confusion matrix, and
the precision and recall. In this dataset **class 0 = malignant** and **class 1 = benign**, and we treat
**malignant as the positive class**, so use `pos_label=0` for precision and recall. The provided code draws the
confusion matrix and a bar chart of the three scores.

**Answer:**

1. In one sentence each, say what **precision** and **recall** mean *for this problem*. (5 pts)
2. This model will be used as a first screening tool. Which kind of mistake is **worse**: calling a malignant
   tumor benign, or calling a benign tumor malignant? Which of precision or recall should the hospital watch
   most closely, and why? Is "98% accurate" by itself enough to decide the model is good enough to use? (10 pts)

The remaining 10 points are for the code and the confusion matrix: it must be computed on the test set, with
malignant as the positive class.

In [ ]:
# --- YOUR CODE (4 lines) -------------------------------------------------------------
# 1. pred_logreg: log_pipe's predictions on Xc_test
# 2. cm: confusion_matrix(actual, predicted) for the test set (rows = actual, columns = predicted)
# 3. precision_mal: precision for malignant (class 0) -> precision_score(..., pos_label=0)
# 4. recall_mal: recall for malignant (class 0)       -> recall_score(..., pos_label=0)


# --- PROVIDED: do not change ------------------------------------------------------
print("Class order:", [str(n) for n in cancer.target_names], "(class 0 first)")
print(f"Precision (malignant): {precision_mal:.3f}")
print(f"Recall    (malignant): {recall_mal:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ConfusionMatrixDisplay(cm, display_labels=cancer.target_names).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix (test set)")

bars = axes[1].bar(["precision\n(malignant)", "recall\n(malignant)", "accuracy"],
                   [precision_mal, recall_mal, logreg_acc], color=["steelblue", "crimson", "lightgray"])
axes[1].bar_label(bars, fmt="%.3f")
axes[1].set(ylim=(0, 1.1), title="Scores for logistic regression")

plt.tight_layout()
plt.show()

**Your answers** (1. what precision and recall mean here; 2. which mistake is worse, which score to watch, and whether accuracy is enough):

---

### AI Check

In 1-2 sentences: identify one line of AI-generated code you used in this assignment (or say you used none) and how you verified it was correct for this specific data — for example, by checking the class order in `y_cancer` or comparing a metric against your own hand calculation.

**Your answer:**

---

## Grading Rubric

| Criterion | Points |
|---|---|
| Deliverable 1 — ran the provided baseline cell; explains why R² is slightly below 0 and what the baseline is for | 10 |
| Deliverable 2 — both models fit on the training data and evaluated on the same test set; interpretation and coefficient/causation answer are sound | 20 |
| Deliverable 3 — split stratified; baseline and accuracy correct; scaling is inside the pipeline; leakage explanation is correct | 15 |
| Deliverable 4 — scaled k values + unscaled k=5 computed correctly; scaling and choosing-k reasoning are correct | 20 |
| Deliverable 5 — confusion matrix, precision and recall computed for the correct (malignant) class; reasoning about which mistake is worse is sound | 25 |
| Notebook runs top to bottom without errors | 10 |
| **Total** | **100** |

---

## Submission

### GitHub (same repo as Assignments 1 and 2)

1. Use the same `cs383-assignments` repo you created for Assignment 1. Inside it, create a new folder:
   `assignment3/`.
2. Put your completed notebook in that folder, keeping its original filename:
   `assignment3_sklearn_models.ipynb`.

```bash
git add -A
git commit -m "Assignment 3: regression and classification models"
git push
```

Submit the **direct link to your notebook file on GitHub** (the URL should end in
`assignment3/assignment3_sklearn_models.ipynb`) on BrightSpace under **Assignment 3**.

Make sure your notebook runs top to bottom without errors before you submit — a notebook that only
works because cells were run out of order will not be graded as passing.